# LogSeer Training
---

Trains any combination of neural network and sklearn models (xgb, svm, rf) to classify server logs as success or failure.

Run cells from top to bottom. Colab-only cells are marked — skip them if running elsewhere.

**Setup**: Place log data under `DATA_DIR` with this structure:

```
DATA_DIR/
├── error/
│   ├── 1/
│   │   ├── log1.txt
│   │   └── log2.txt
│   └── 2/
│       └── log1.txt
└── success/
    ├── 1/
    │   └── log1.txt
    └── 2/
        └── log1.txt
```

Each subdirectory under `error/` and `success/` is one operation run. Adjust the Configuration cell as needed — key knobs are `SUCCESS_LOG_RATIO` (class balance), `NN_ERROR_WEIGHT` / `SKLEARN_WEIGHT` (error-class emphasis), `SKLEARN_MODELS` (e.g. `'xgb,svm'`), and `TEST_NN` (set to `False` to skip the NN entirely).

**Output**: saves `logseer.keras`, `tokenizer.pickle`, and one `.pkl` per sklearn model to the repo root. Run the last cell to copy them to Google Drive.

> **▶️ Start here** — run all cells from top to bottom. Skip any marked **COLAB ONLY** if running on a self-hosted environment.

In [ ]:
# Setup
import os, sys

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

if not os.path.exists('logseer'):
    os.system('git clone https://github.com/masahiko-shibata/logseer.git')
    os.chdir('logseer')

sys.path.insert(0, '.')

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
from logseer import *

if not tf.config.list_physical_devices('GPU'):
    print("⚠️ WARNING: No GPU detected! Will run on CPU. Nural network training will be very slow. If you are on Colab, consider changing Runtime > Change runtime type > Hardware accelerator.")
else:
    print("✅ GPU is available:", tf.config.list_physical_devices('GPU'))

---
### ☁️ COLAB ONLY — run the cell below if you want to load data from Google Drive

> Load training data from a zip on Google Drive. Default: `Colab Notebooks/logseer/data/data.zip` in My Drive. Zip the `data/` folder itself so it extracts to `data/` — don't zip the contents directly. **Skip if running on a self-hosted environment** and copy data to `DATA_DIR` manually.

---

In [ ]:
# Data copy from Google Drive
from google.colab import drive
import shutil, zipfile
drive.mount('/content/drive')

!rm -rf logs data.zip 2>/dev/null
shutil.copy('/content/drive/MyDrive/Colab Notebooks/logseer/data/data.zip', 'data.zip')
with zipfile.ZipFile('data.zip', 'r') as z:
    z.extractall('.')
print(os.listdir('./'))

---
### Configuration basics

> **`REPETITION`** — training is run this many times, each on a different random split of the data. This is primarily for tuning — more repetitions give more stable performance estimates, making it easier to compare configurations. The default `100` works well when your error set is small (tens of log sets). With thousands of error log sets, `5`–`10` should be enough — larger datasets vary less between repetitions.
>
> **`TEST_NN`** — set to `False` to skip neural network training entirely and run only the sklearn models defined in `SKLEARN_MODELS`. Sklearn training is CPU-friendly and finishes in seconds, making it useful for rapid iteration when you don't have a GPU or want quick feedback.
>
> **Final training to produce production models** — once you're done tuning, run one last time with `VALIDATE_ON_TEST_DATA = True` and `REPETITION = 1`. This trains on the full dataset and produces model files for production use.

---

In [ ]:
# Configuration

# Paths
DATA_DIR            = 'data'
MODEL_SAVE_PATH     = 'logseer.keras'
TOKENIZER_PATH      = 'tokenizer.pickle'

# Data
NUM_CHAR            = 3000
TO_ID               = 6000
VALIDATION_SPLIT      = 0.1
VALIDATE_ON_TEST_DATA = False
SUCCESS_LOG_RATIO     = 99

# Model
MODEL_NAME          = 'LogCNNv2'
EMBEDDING_LAYER     = 'vanilla'
EMBEDDING_DIM       = 32
MAX_NB_WORDS        = 24000
MAX_SEQUENCE_LENGTH = 26000

# Training
EPOCHS              = 60
BATCH_SIZE          = 32
LEARNING_RATE       = 0.0003
REPETITION          = 100
RETRAIN             = False
NN_ERROR_WEIGHT     = 2
LABEL_SMOOTHING     = 0.1
TEST_NN             = True   # set to False to skip NN training and run sklearn only

# Early stopping
USE_EARLY_STOPPING   = True
PATIENCE             = 15
RESTORE_BEST_WEIGHTS = False
ES_START_FROM_EPOCH  = 10
MONITOR              = 'val_f1'
MODE                 = 'max'

# Checkpoint: 'multi_metric' | 'best_f1' | 'standard'
CHECKPOINT_TYPE      = 'best_f1'
START_FROM_EPOCH     = 0
MAX_LOSS            = 0.7

# Sklearn models to train alongside the NN: 'xgb', 'svm', 'rf' (comma-separated string or list)
SKLEARN_MODELS      = 'xgb'
SKLEARN_WEIGHT      = 6

---
> At this point you have loaded your data and set the configuration. The next cell runs the training loop — it will sample the data, train the model(s), and print evaluation metrics after each repetition.
---

In [ ]:
# Training loop
tester = run_training(
    DATA_DIR,
    max_nb_words=MAX_NB_WORDS,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
    embedding_dim=EMBEDDING_DIM,
    validation_split=VALIDATION_SPLIT,
    validate_on_test_data=VALIDATE_ON_TEST_DATA,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    model_save_path=MODEL_SAVE_PATH,
    tokenizer_path=TOKENIZER_PATH,
    embedding_layer_type=EMBEDDING_LAYER,
    model_name=MODEL_NAME,
    repetition=REPETITION,
    nn_error_weight=NN_ERROR_WEIGHT,
    label_smoothing=LABEL_SMOOTHING,
    learning_rate=LEARNING_RATE,
    max_loss=MAX_LOSS,
    retrain=RETRAIN,
    numchar=NUM_CHAR,
    toid=TO_ID,
    checkpoint_type=CHECKPOINT_TYPE,
    start_from_epoch=START_FROM_EPOCH,
    es_start_from_epoch=ES_START_FROM_EPOCH,
    use_early_stopping=USE_EARLY_STOPPING,
    patience=PATIENCE,
    monitor=MONITOR,
    mode=MODE,
    restore_best_weights=RESTORE_BEST_WEIGHTS,
    test_nn=TEST_NN,
    sklearn_models=SKLEARN_MODELS,
    sklearn_weight=SKLEARN_WEIGHT,
)

---
### ☁️ COLAB ONLY — run the cell below to save model files to Google Drive

> Save trained model files to `Colab Notebooks/logseer/models` in your My Drive. **Skip if running on a self-hosted environment** — models are already saved to the repo root.

---

In [ ]:
# Copy model files to Google Drive
import shutil
from google.colab import drive
drive.mount('/content/drive')
DRIVE_MODEL_DIR = '/content/drive/MyDrive/Colab Notebooks/logseer/models'
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
shutil.copy(MODEL_SAVE_PATH, DRIVE_MODEL_DIR + '/' + MODEL_SAVE_PATH)
shutil.copy(TOKENIZER_PATH,  DRIVE_MODEL_DIR + '/' + TOKENIZER_PATH)
for m in [m.strip() for m in SKLEARN_MODELS.replace(',', ' ').split() if m.strip()]:
    shutil.copy(f'{m}.pkl', DRIVE_MODEL_DIR + f'/{m}.pkl')
print('Models saved to', DRIVE_MODEL_DIR)